# 11 · Exporter and model-family comparison on the board

Decision trees can also be converted to C with emlearn or micromlgen, and earlier ESP32 IDS work deployed shallow trees and boosted models.
This notebook builds a controlled comparison and, after the variants are flashed, analyses the logs.

All variants share the same test vectors and the same benchmark sketch (it only calls `predict_esp32_dt()`), so only the generated model code differs.

| Part | Variants | What differs |
|---|---|---|
| **A · Exporters** (same production tree) | `prod_ours` · `prod_emlearn` · `prod_micromlgen` | the exporter: FaithTree arrays vs inline if-else code (emlearn, micromlgen) |
| **B · Model families** (same training rows) | `fam_tree` · `fam_depth5` (shallow tree) · `fam_forest` (random forest, hard vote) · `fam_forest_emlearn` | accuracy vs flash vs latency |

**Notes**
* scikit-learn's forest `predict` is a soft vote; the exported C code is a hard majority vote. The Python reference for forests is therefore the hard vote, and the forest MCC is the hard-vote MCC.
* emlearn's default int16 features would truncate the scaled inputs, so it is run in float mode.
* Host-side sizes (x86-64, `-Os`) are a proxy for relative comparison only; firmware size and latency come from the board logs.

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default
from esp32_export import (export_tree_pipeline_to_c, create_test_vectors_header, write_benchmark_sketch, select_validation_samples)
import toolchain_export as tc, paper_utils as pu
from sklearn.ensemble import RandomForestClassifier
import shutil, time
try:
    tc.import_emlearn_safely(); HAS_EMLEARN = True       # NOTE: a plain `import emlearn` sets numpy.seterr(all='raise') and breaks reading CSVs with float32 overflow
except Exception: HAS_EMLEARN = False
try:
    import micromlgen; HAS_MICROMLGEN = True
except Exception: HAS_MICROMLGEN = False
print("Dataset:", ACTIVE_DATASET, "| emlearn:", HAS_EMLEARN, "| micromlgen:", HAS_MICROMLGEN, "| g++:", tc._cxx())
print(f"vectors per variant {TC_N_SAMPLES} | rows for the model-family comparison {TC_TRAIN_ROWS:,} | forest trees {TC_FOREST_TREES}")

## 1 · Data, production tree, model-family training rows

In [ ]:
artifact, metadata = load_artifact_and_metadata()
X, y, le, numeric_cols = load_publishability_data()
class_names = list(le.classes_)
X_train, X_val, X_test, y_train, y_val, y_test, _, _ = same_split(X, y)
rare_classes, extremely_rare_classes = infer_rare_classes(y_train)
BASE_PARAMS, BASE_K = dict(metadata["best_params"]), min(int(metadata["best_k"]), len(numeric_cols))
Xtr, ytr = stratified_sample(X_train, y_train, TC_TRAIN_ROWS, SEED)       # SAME rows for every model-family variant
Xe, ye = stratified_sample(X_test, y_test, TC_EVAL_ROWS, SEED)            # test rows for MCC
print("production model:", Path(metadata["model_path"]).name, "| K =", BASE_K, "| params:", BASE_PARAMS)
print(f"model-family training rows {len(Xtr):,} | MCC computed on {len(Xe):,} test rows")

forest_kw = {k: v for k, v in BASE_PARAMS.items() if k in ("criterion", "max_depth", "min_samples_leaf", "min_samples_split", "ccp_alpha")}
frac = min(1.0, 1_000_000 / max(len(Xtr), 1))
def fit_art(clf, k=BASE_K):
    p = build_safe_pipeline(clf, k=k); p.fit(Xtr, ytr); return {"pipeline": p, "label_encoder": le, "numeric_cols": numeric_cols}
t0 = time.perf_counter()
art_tree   = fit_art(DecisionTreeClassifier(random_state=SEED, **{k: v for k, v in BASE_PARAMS.items() if k in forest_kw or k == "class_weight"}))
art_depth5 = fit_art(DecisionTreeClassifier(max_depth=5, random_state=SEED))
art_forest = fit_art(RandomForestClassifier(n_estimators=TC_FOREST_TREES, bootstrap=True, max_samples=frac if frac < 1 else None, n_jobs=-1, random_state=SEED, **forest_kw))
print(f"trained in {time.perf_counter() - t0:.0f}s")

## 2 · Define the variants

In [ ]:
def n_nodes(art):
    c = art["pipeline"].named_steps["clf"]
    return sum(int(e.tree_.node_count) for e in c.estimators_) if hasattr(c, "estimators_") else int(c.tree_.node_count)

VAR = {}      # name -> dict(art, builder, predict, group, family)
prod_clf = artifact["pipeline"].named_steps["clf"]
VAR["prod_ours"] = dict(group="A toolchain", art=artifact, build=lambda a, d: export_tree_pipeline_to_c(a, d), pred=lambda a, X: a["pipeline"].predict(X))
if HAS_EMLEARN:
    code_, call_, decl_ = tc.emlearn_code(prod_clf)
    VAR["prod_emlearn"] = dict(group="A toolchain", art=artifact, build=lambda a, d, c=code_, k=call_, dd=decl_: tc.export_external_classifier_to_c(a, d, c, k, n_nodes(a), dd), pred=lambda a, X: a["pipeline"].predict(X))
if HAS_MICROMLGEN:
    code_, call_, decl_ = tc.micromlgen_code(prod_clf)
    VAR["prod_micromlgen"] = dict(group="A toolchain", art=artifact, build=lambda a, d, c=code_, k=call_, dd=decl_: tc.export_external_classifier_to_c(a, d, c, k, n_nodes(a), dd), pred=lambda a, X: a["pipeline"].predict(X))
VAR["fam_tree"]   = dict(group="B model family", art=art_tree,   build=lambda a, d: export_tree_pipeline_to_c(a, d), pred=lambda a, X: a["pipeline"].predict(X))
VAR["fam_depth5"] = dict(group="B model family", art=art_depth5, build=lambda a, d: export_tree_pipeline_to_c(a, d), pred=lambda a, X: a["pipeline"].predict(X))
VAR["fam_forest"] = dict(group="B model family", art=art_forest, build=lambda a, d: tc.export_forest_to_c(a, d), pred=lambda a, X: tc.forest_hardvote_predict(a["pipeline"], X))
if HAS_EMLEARN:
    rf = art_forest["pipeline"].named_steps["clf"]; code_, call_, decl_ = tc.emlearn_code(rf)
    VAR["fam_forest_emlearn"] = dict(group="B model family", art=art_forest, build=lambda a, d, c=code_, k=call_, dd=decl_: tc.export_external_classifier_to_c(a, d, c, k, n_nodes(a), dd), pred=lambda a, X: tc.forest_hardvote_predict(a["pipeline"], X))
print("variants:", list(VAR))

## 3 · Accuracy of each variant (test rows) and generation of the board projects

In [ ]:
BASE = ESP32_DIR / "toolchain_compare"; BASE.mkdir(parents=True, exist_ok=True)
sel = select_validation_samples(y_test, TC_N_SAMPLES, "class_balanced", SEED)
X_sample, y_sample = X_test.iloc[sel], np.asarray(y_test)[sel]
rows = []
for name, v in VAR.items():
    d = BASE / name / "esp32_dt_test"; d.mkdir(parents=True, exist_ok=True)
    v["dir"] = d
    pe = v["pred"](v["art"], Xe); me = compute_enhanced_metrics(ye, pe, rare_classes, extremely_rare_classes)
    v["py_sample"] = v["pred"](v["art"], X_sample)
    v["build"](v["art"], d)
    create_test_vectors_header(X_sample, y_sample, v["py_sample"], v["art"], d, sample_ids=X_sample.index.values)
    write_benchmark_sketch(d)
    rows.append({"variant": name, "group": v["group"], "test_mcc": me["mcc"], "test_macro_f1": me["macro_f1"], "test_accuracy": me["accuracy"],
                 "nodes": n_nodes(v["art"]), "trees": len(getattr(v["art"]["pipeline"].named_steps["clf"], "estimators_", [1])),
                 "model_header_kb": (d / "esp32_dt_model.h").stat().st_size / 1024})
acc = pd.DataFrame(rows); display(acc.round(4))

## 4 · Host verification (no hardware): does every generated C++ program reproduce its Python reference?
Also a **relative** size proxy (code + read-only data of the model alone, x86-64 `-Os`). It is *not* the ESP32 flash size.

In [ ]:
vrows = []
for name, v in VAR.items():
    chk = tc.host_cpp_check(v["dir"], v["py_sample"]); fp = tc.host_footprint(v["dir"])
    vrows.append({"variant": name, "compiled": bool(chk and chk.get("compiled")), "n_vectors": chk.get("n") if chk else None, "agreement_host": chk.get("agreement") if chk else None,
                  "host_code+rodata_bytes_proxy": fp})
    print(f"{name:20s} compiled={vrows[-1]['compiled']}  agreement={vrows[-1]['agreement_host']}  host size proxy={fp} B", flush=True)
host = pd.DataFrame(vrows); res = acc.merge(host, on="variant"); res.to_csv(BASE / "toolchain_host_check.csv", index=False)
fe = tc.emulate_c_forest_predict(art_forest, X_test.iloc[:min(len(X_test), 500_000)]); fr = tc.forest_hardvote_predict(art_forest["pipeline"], X_test.iloc[:min(len(X_test), 500_000)])
print(f"\nforest float32 C-logic emulation vs hard-vote reference on {len(fe):,} test rows: {float((fe == fr).mean()):.5f}")
print(f"(sklearn's soft-vote predict vs the hard vote agree on only {float((art_forest['pipeline'].predict(X_test.iloc[:len(fe)]) == fr).mean()):.4f} of rows -> the hard vote is a slightly different classifier)")
display(res.round(4))

## 5 · Flash the variants
For each variant folder printed below: open `esp32_dt_test.ino` in the Arduino IDE, upload, wait for `#DONE` and save the whole Serial output as
`esp32/serial_logs/toolchain_<variant>.txt`. Use the same board, IDE/core settings and Serial speed (115200) for all variants.

In [ ]:
print("Variant folders to flash:\n")
for name, v in VAR.items():
    print(f"  {name:20s} -> {v['dir']}\n  {'':20s}    save log as  esp32\\serial_logs\\toolchain_{name}.txt")
json.dump({n: str(v["dir"]) for n, v in VAR.items()}, open(BASE / "toolchain_manifest.json", "w"), indent=2)

## 6 · Analysis of the logs  *(re-run this cell after flashing)*

In [ ]:
LOGS = ROOT / "esp32" / "serial_logs"; FIG, TAB = ROOT / "paper" / "figures", ROOT / "paper" / "tables"; FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
hw = []
for name in VAR:
    f = LOGS / f"toolchain_{name}.txt"
    if not f.exists(): continue
    log = pu.parse_serial_log(f)
    if log["samples"].empty or log["legacy"]: print(f"  {name}: log found but not in the benchmark-sketch format -> skipped"); continue
    r = pu.summarize_hardware(ACTIVE_DATASET, log); r["variant"] = name; hw.append(r)
print("logs found for:", [h["variant"] for h in hw] or "none yet")
if hw:
    H = pd.DataFrame(hw)
    for c in H.columns:
        if c not in ("variant", "dataset", "log_format", "chip"): H[c] = pd.to_numeric(H[c], errors="coerce")
    keep = ["variant", "python_esp32_matches", "n_samples", "agreement", "mean_latency_us_batch", "median_us", "p95_us", "max_us", "benchmark_firmware_bytes", "flash_used_pct", "flash_headroom_bytes"]
    T = res.merge(H[[c for c in keep if c in H.columns]], on="variant", how="left")
    ref = T.loc[T.variant == "prod_ours", "benchmark_firmware_bytes"]
    if len(ref) and pd.notna(ref.iloc[0]): T["firmware_delta_vs_prod_ours_bytes"] = T["benchmark_firmware_bytes"] - float(ref.iloc[0])
    T.to_csv(TAB / "tc_toolchain_comparison.csv", index=False); display(T.round(3))
    pu.save_table(T[[c for c in ["variant", "group", "test_mcc", "agreement", "mean_latency_us_batch", "p95_us", "benchmark_firmware_bytes", "firmware_delta_vs_prod_ours_bytes"] if c in T.columns]],
                  TAB / "tc_toolchain_comparison_paper", "Toolchain and model-family comparison on the ESP32 (same preprocessing, same vectors).", "tab:toolchain", "{:.3f}")
    pu.set_paper_style(); good = T.dropna(subset=["mean_latency_us_batch"])
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for _, r in good.iterrows():
        ax.scatter(r["mean_latency_us_batch"], r["test_mcc"], s=60 + 0.00018 * float(r["benchmark_firmware_bytes"]), alpha=.7, color="#1f77b4" if r["group"].startswith("A") else "#d95f02")
        ax.annotate(r["variant"], (r["mean_latency_us_batch"], r["test_mcc"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("mean on-device latency per prediction (µs, log)"); ax.set_ylabel("test MCC")
    ax.set_title("accuracy vs latency (bubble = firmware size);  blue = toolchains, orange = model families")
    pu.save_fig(fig, FIG / "tc_tradeoff"); plt.show()

## 7 · Pareto summary

In [ ]:
def dominated(r, T):
    # r is dominated if another variant is at least as good on MCC, latency and firmware, and strictly better on one
    for _, o in T.iterrows():
        if o["variant"] == r["variant"]: continue
        ge = (o["test_mcc"] >= r["test_mcc"] - 1e-9) and (o["mean_latency_us_batch"] <= r["mean_latency_us_batch"]) and (o["benchmark_firmware_bytes"] <= r["benchmark_firmware_bytes"])
        gt = (o["test_mcc"] > r["test_mcc"] + 1e-9) or (o["mean_latency_us_batch"] < r["mean_latency_us_batch"]) or (o["benchmark_firmware_bytes"] < r["benchmark_firmware_bytes"])
        if ge and gt: return o["variant"]
    return None

if hw and {"mean_latency_us_batch", "benchmark_firmware_bytes"} <= set(T.columns) and T["mean_latency_us_batch"].notna().sum() >= 2:
    G = T.dropna(subset=["mean_latency_us_batch", "benchmark_firmware_bytes"]).copy()
    print("Pareto analysis over (higher MCC, lower latency, smaller firmware):\n")
    for _, r in G.iterrows():
        dom = dominated(r, G); print(f"  {r['variant']:20s} {'PARETO-OPTIMAL' if dom is None else 'dominated by ' + dom}")
    A = G[G.group.str.startswith("A")]
    if "prod_ours" in set(A.variant) and len(A) > 1:
        o = A[A.variant == "prod_ours"].iloc[0]
        for _, r in A[A.variant != "prod_ours"].iterrows():
            print(f"\n  vs {r['variant']}: FaithTree export is {o['mean_latency_us_batch'] / r['mean_latency_us_batch'] - 1:+.1%} latency and {(o['benchmark_firmware_bytes'] - r['benchmark_firmware_bytes']) / 1024:+.1f} KB firmware; agreement {o['agreement']:.4f} vs {r['agreement']:.4f}")
else:
    print("No (or too few) board logs yet -> only the host verification above is available. Flash the variants (section 5) and re-run sections 6-7.")